In [ ]:
with open("dataset/processed/corpus_clean.txt", "r") as f:
    text = f.read()

characters = sorted(list(set(text)))
vocab_size = len(characters)
char_to_idx = { ch:i for i,ch in enumerate(characters) }
idx_to_char = { i:ch for i,ch in enumerate(characters) }
encode = lambda xs: [char_to_idx[x] for x in xs]
decode = lambda xs: ''.join([idx_to_char[x] for x in xs])

In [ ]:
import torch

data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100])

# Visualize

Lets visualize the frequency of the token-pairs

In [ ]:
counts = {}
for ch1, ch2 in zip(data, data[1:]):
    bigram = (ch1.item(), ch2.item())
    counts[bigram] = counts.get(bigram, 0) + 1

decoded_counts = {decode(bigram): count for bigram, count in counts.items()}
sorted(decoded_counts.items(), key=lambda x: x[1], reverse=True)[:50]

In [ ]:
img = torch.zeros(vocab_size, vocab_size, dtype=torch.long)
for (i, j), count in counts.items():
    img[i, j] = count

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

plt.figure(figsize=(32,32))
plt.imshow(img, cmap='Blues', norm=LogNorm(vmin=1e-3, vmax=img.max()))
for i in range(vocab_size):
    for j in range(vocab_size):
        chstr = decode([i, j])
        chstr = chstr.replace("$", "\$")
        plt.text(j, i, chstr, ha="center", va="bottom", color='gray')
        plt.text(j, i, img[i, j].item(), ha="center", va="top", color='gray')
plt.axis('off')

# Data loader

Generator of pairs of inputs (X) and targets (Y)

X - n-token sequence
Y - following token

for the bigram, the working n is 1 (which is called context size), but we can feed arbitrary n for compatibility with future sessions

In [ ]:
n = int(len(text) * 0.9)
train_data = data[:n]
val_data = data[n:]

In [ ]:
context_size = 8
train_data[:context_size+1]

In [ ]:
x = train_data[:context_size]
y = train_data[1:context_size+1]
for t in range(context_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

In [ ]:
torch.manual_seed(1111)
batch_size = 4
context_size = 8

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - context_size, (batch_size,))
    x = torch.stack([data[i:i+context_size] for i in ix])
    y = torch.stack([data[i+1:i+context_size+1] for i in ix])
    return x, y

In [ ]:
xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('---')
print('targets:')
print(yb.shape)
print(yb)

# Bi-gram torch

Torch implementation of Bi-Gram generative model. It's essentialy learned version of the table above.

It's nn.Module so its going to be compatible with future training and eval infrastructure.

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F

In [ ]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx) # (batch_size, context_size, vocab_size)

        if targets is not None:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        else:
            loss = None

        return logits, loss

In [ ]:
def generate(model, start_idx, number_of_tokens):
    idx = start_idx
    for _ in range(number_of_tokens):
        logits, loss = model(idx)
        logits = logits[:, -1, :] # (batch_size, vocab_size)
        probs = F.softmax(logits, dim=1)
        idx_next = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

In [ ]:
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
logits.shape, loss

In [ ]:
start_idx = torch.zeros((1, 1), dtype=torch.long)
max_tokens = 300
print(decode(
    generate(m, start_idx=start_idx, number_of_tokens=max_tokens)[0].tolist()
))

In [ ]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [ ]:
batch_size = 32

report_frequency = 1000

for steps in range(10000):
    xb, yb = get_batch('train')

    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if steps % report_frequency == 0:
        print(f"Step {steps}, loss: {loss.item()}")

print(loss.item())

In [ ]:
start_idx = torch.zeros((1, 1), dtype=torch.long)
max_tokens = 300
print(decode(
    generate(m, start_idx=start_idx, number_of_tokens=max_tokens)[0].tolist()
))

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
img = m.token_embedding_table.weight.detach().cpu().numpy()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

plt.figure(figsize=(32,32))
plt.imshow(img, cmap='Blues')
for i in range(vocab_size):
    for j in range(vocab_size):
        chstr = decode([i, j])
        chstr = chstr.replace("$", "\$")
        plt.text(j, i, chstr, ha="center", va="bottom", color='gray')
plt.axis('off')